# C9-dimensionality-reduction — Practice p24

**Type:** challenge · **Difficulty:** advanced · **Concepts:** numpy-pca-class-from-scratch, pca-black-box-insufficiency

Implement the complete p23 `NumpyPCA` contract and the function
`subspace_projector(model)`.
The function accepts a fitted `NumpyPCA` and returns
`model.components_.T @ model.components_` as a finite float square array.
It must reject an unfitted model with `ValueError`.

This challenge grades two degenerate regimes.

1. A rank-two centered matrix in four features: fitting two components must reconstruct exactly up to `ATOL`, and a four-component fit must report two trailing zero variances.
2. A matrix with a repeated largest covariance eigenvalue: any orthonormal basis of the top-two eigenspace is valid, so the checker compares only subspace projectors.

Do not sign-fix, sort, or compare repeated component rows against a hidden row order.
The reusable class still uses sample covariance with denominator `n - 1`, rejects non-finite covariance before decomposition, makes exactly one `np.linalg.eigh` call per successful fit, rejects non-finite returned eigenpairs, uses full-spectrum explained ratios, and satisfies the validation/data-flow contract from p22–p23.
Ratio normalization must scale the full nonnegative spectrum by its maximum before summing, with an all-zero ratio vector only when that maximum is zero.
Fit is atomic: any failed first fit leaves the model unfitted, and any failed refit preserves all four arrays from the previous successful fit.

**Zero points:** sklearn PCA, scipy PCA, or helpers that invoke them.
Use `ATOL = 1e-10`, `RTOL = 0.0`; do not mutate inputs.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0


class NumpyPCA:
    def __init__(self, n_components):
        self.n_components = n_components

    @staticmethod
    def _numeric_matrix(value, *, allow_empty_rows=False):
        try:
            array = np.asarray(value)
        except (TypeError, ValueError):
            raise ValueError("input must be a numeric matrix") from None
        if array.ndim != 2:
            raise ValueError("input must be a matrix")
        if not np.issubdtype(array.dtype, np.number) or np.issubdtype(array.dtype, np.complexfloating):
            raise ValueError("input must contain real numeric values")
        if not allow_empty_rows and array.shape[0] == 0:
            raise ValueError("input must contain at least one row")
        result = np.array(array, dtype=float, copy=True)
        if not np.isfinite(result).all():
            raise ValueError("input must contain only finite values")
        return result

    def fit(self, X):
        array = self._numeric_matrix(X)
        n, d = array.shape
        if n < 2 or d < 1:
            raise ValueError("fit requires at least two rows and one feature")
        k = self.n_components
        if isinstance(k, (bool, np.bool_)) or not isinstance(k, (int, np.integer)):
            raise ValueError("n_components must be an integer")
        if not 1 <= int(k) <= d:
            raise ValueError("n_components is outside the feature range")

        with np.errstate(over="ignore", invalid="ignore"):
            mean = array.mean(axis=0)
            centered = array - mean
            covariance = centered.T @ centered / (n - 1)
        if not np.isfinite(mean).all() or not np.isfinite(centered).all():
            raise ValueError("centering produced non-finite values")
        if not np.isfinite(covariance).all():
            raise ValueError("covariance must contain only finite values")

        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        if not np.isfinite(eigenvalues).all() or not np.isfinite(eigenvectors).all():
            raise ValueError("eigendecomposition returned non-finite values")
        order = np.argsort(eigenvalues)[::-1]
        full_variances = np.maximum(eigenvalues[order], 0.0)
        ordered_vectors = eigenvectors[:, order]
        scale = full_variances.max()
        k = int(k)

        candidate_mean = np.array(mean, dtype=float, copy=True)
        candidate_components = np.array(ordered_vectors[:, :k].T, dtype=float, copy=True)
        candidate_variances = np.array(full_variances[:k], dtype=float, copy=True)
        if scale == 0.0:
            ratios = np.zeros(k, dtype=float)
        else:
            scaled = full_variances / scale
            denominator = scaled.sum()
            ratios = scaled[:k] / denominator
        candidate_ratios = np.array(ratios, dtype=float, copy=True)
        if not all(np.isfinite(value).all() for value in (
            candidate_mean,
            candidate_components,
            candidate_variances,
            candidate_ratios,
        )):
            raise ValueError("learned state must contain only finite values")

        self.mean_ = candidate_mean
        self.components_ = candidate_components
        self.explained_variance_ = candidate_variances
        self.explained_variance_ratio_ = candidate_ratios
        return self

    def transform(self, X):
        if not hasattr(self, "components_"):
            raise ValueError("fit must be called before transform")
        array = self._numeric_matrix(X)
        if array.shape[1] != self.mean_.shape[0]:
            raise ValueError("feature count does not match fitted data")
        return (array - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    def inverse_transform(self, Z):
        if not hasattr(self, "components_"):
            raise ValueError("fit must be called before inverse_transform")
        scores = self._numeric_matrix(Z)
        if scores.shape[1] != self.components_.shape[0]:
            raise ValueError("score width does not match retained components")
        return scores @ self.components_ + self.mean_


def subspace_projector(model):
    if not isinstance(model, NumpyPCA) or not hasattr(model, "components_"):
        raise ValueError("a fitted NumpyPCA is required")
    projector = model.components_.T @ model.components_
    if not np.issubdtype(projector.dtype, np.floating) or not np.isfinite(projector).all():
        raise ValueError("fitted components must be finite floats")
    return projector


## Immutable contract check — do not edit

The repeated fixture deliberately permits different signs, row order, and rotations.
Only covariance spectra, projectors, transformations, and reconstructions are compared.


In [ ]:
import dis
import inspect
import types

_ORIGINAL_EIGH_P24 = np.linalg.eigh


def _audit_p24():
    pending = [subspace_projector]
    for value in vars(NumpyPCA).values():
        if isinstance(value, (staticmethod, classmethod)):
            value = value.__func__
        if isinstance(value, types.FunctionType):
            pending.append(value)
    seen = set()
    while pending:
        function = pending.pop()
        if id(function) in seen:
            continue
        seen.add(id(function))
        codes = [function.__code__]
        while codes:
            code = codes.pop()
            codes.extend(item for item in code.co_consts if isinstance(item, types.CodeType))
            assert not ({"sklearn", "scipy"} & {name.lower() for name in code.co_names})
            for instruction in dis.get_instructions(code):
                if instruction.opname in {"IMPORT_NAME", "IMPORT_FROM"}:
                    assert not str(instruction.argval).lower().startswith(("sklearn", "scipy"))
            for name in code.co_names:
                value = function.__globals__.get(name)
                if isinstance(value, types.FunctionType):
                    pending.append(value)
                assert not str(getattr(value, "__module__", "")).lower().startswith(("sklearn", "scipy"))
    try:
        source = inspect.getsource(NumpyPCA).lower() + inspect.getsource(subspace_projector).lower()
    except (OSError, TypeError):
        source = ""
    assert "sklearn" not in source and "scipy" not in source


_audit_p24()

try:
    subspace_projector(NumpyPCA(1))
except ValueError:
    pass
else:
    raise AssertionError("unfitted projector request must raise ValueError")

for _prefit_method_p24, _prefit_arg_p24 in (
    ("transform", np.ones((2, 4))),
    ("inverse_transform", np.ones((2, 1))),
):
    try:
        getattr(NumpyPCA(1), _prefit_method_p24)(_prefit_arg_p24)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{_prefit_method_p24} before fit must raise ValueError")

_latent_p24 = np.array([
    [-3.0, 1.0], [-1.0, -2.0], [0.0, 1.0],
    [1.0, 3.0], [2.0, -1.0], [1.0, -2.0],
])
_mix_p24 = np.array([[1.0, 2.0, -1.0, 0.5], [0.0, 1.0, 2.0, -1.0]])
_X_rank_p24 = _latent_p24 @ _mix_p24 + np.array([7.0, -3.0, 2.0, 5.0])
assert np.linalg.matrix_rank(_X_rank_p24 - _X_rank_p24.mean(axis=0)) == 2
_X_rank_before_p24 = _X_rank_p24.copy()
_eigh_calls_p24 = []
def _traced_eigh_p24(matrix):
    _eigh_calls_p24.append(np.array(matrix, copy=True))
    return _ORIGINAL_EIGH_P24(matrix)
np.linalg.eigh = _traced_eigh_p24
_rank_model_p24 = NumpyPCA(2)
_fit_calls_p24 = []
_original_rank_fit_p24 = _rank_model_p24.fit
def _traced_rank_fit_p24(X):
    _fit_calls_p24.append(np.array(X, copy=True))
    return _original_rank_fit_p24(X)
_rank_model_p24.fit = _traced_rank_fit_p24
_Z_rank_p24 = _rank_model_p24.fit_transform(_X_rank_p24)
_Xhat_rank_p24 = _rank_model_p24.inverse_transform(_Z_rank_p24)
_P_rank_p24 = subspace_projector(_rank_model_p24)
assert np.array_equal(_X_rank_p24, _X_rank_before_p24)
assert isinstance(_P_rank_p24, np.ndarray) and _P_rank_p24.shape == (4, 4)
assert np.issubdtype(_P_rank_p24.dtype, np.floating) and np.isfinite(_P_rank_p24).all()
assert np.allclose(_P_rank_p24, _P_rank_p24.T, atol=ATOL, rtol=RTOL)
assert np.allclose(_P_rank_p24 @ _P_rank_p24, _P_rank_p24, atol=ATOL, rtol=RTOL)
assert np.isclose(np.trace(_P_rank_p24), 2.0, atol=ATOL, rtol=RTOL)
_C_rank_p24 = (_X_rank_p24 - _X_rank_p24.mean(axis=0)).T @ (_X_rank_p24 - _X_rank_p24.mean(axis=0)) / 5
assert len(_fit_calls_p24) == 1 and len(_eigh_calls_p24) == 1
assert np.allclose(_eigh_calls_p24[0], _C_rank_p24, atol=ATOL, rtol=RTOL)
_evals_rank_p24, _evecs_rank_p24 = _ORIGINAL_EIGH_P24(_C_rank_p24)
_order_rank_p24 = np.argsort(_evals_rank_p24)[::-1]
_evals_rank_p24 = np.maximum(_evals_rank_p24[_order_rank_p24], 0.0)
_Q_rank_ref_p24 = _evecs_rank_p24[:, _order_rank_p24[:2]].T
_P_rank_ref_p24 = _Q_rank_ref_p24.T @ _Q_rank_ref_p24
for _name_p24, _shape_p24 in (
    ("mean_", (4,)), ("components_", (2, 4)),
    ("explained_variance_", (2,)), ("explained_variance_ratio_", (2,)),
):
    _value_p24 = getattr(_rank_model_p24, _name_p24)
    assert isinstance(_value_p24, np.ndarray) and _value_p24.shape == _shape_p24
    assert np.issubdtype(_value_p24.dtype, np.floating) and np.isfinite(_value_p24).all()
assert np.array_equal(_rank_model_p24.mean_, _X_rank_p24.mean(axis=0))
assert np.allclose(_rank_model_p24.explained_variance_, _evals_rank_p24[:2], atol=ATOL, rtol=RTOL)
assert np.allclose(
    _rank_model_p24.explained_variance_ratio_,
    _evals_rank_p24[:2] / _evals_rank_p24.sum(), atol=ATOL, rtol=RTOL,
)
assert np.allclose(_P_rank_p24, _P_rank_ref_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(_Xhat_rank_p24, _X_rank_p24, atol=ATOL, rtol=RTOL)

_rank_full_p24 = NumpyPCA(4).fit(_X_rank_p24)
assert len(_eigh_calls_p24) == 2
assert np.allclose(_eigh_calls_p24[1], _C_rank_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(_rank_full_p24.explained_variance_, _evals_rank_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(
    _rank_full_p24.explained_variance_ratio_,
    _evals_rank_p24 / _evals_rank_p24.sum(), atol=ATOL, rtol=RTOL,
)
assert np.allclose(_rank_full_p24.explained_variance_[2:], 0.0, atol=ATOL, rtol=RTOL)
assert np.isclose(_rank_full_p24.explained_variance_ratio_.sum(), 1.0, atol=ATOL, rtol=RTOL)

_X_repeat_p24 = np.array([
    [np.sqrt(2.0), 0.0, 0.0, 0.0],
    [-np.sqrt(2.0), 0.0, 0.0, 0.0],
    [0.0, np.sqrt(2.0), 0.0, 0.0],
    [0.0, -np.sqrt(2.0), 0.0, 0.0],
])
_X_repeat_before_p24 = _X_repeat_p24.copy()
_Xc_repeat_p24 = _X_repeat_p24 - _X_repeat_p24.mean(axis=0)
_C_repeat_p24 = _Xc_repeat_p24.T @ _Xc_repeat_p24 / (_X_repeat_p24.shape[0] - 1)
_repeat_model_p24 = NumpyPCA(2).fit(_X_repeat_p24)
np.linalg.eigh = _ORIGINAL_EIGH_P24
assert len(_eigh_calls_p24) == 3
assert np.array_equal(_X_repeat_p24, _X_repeat_before_p24)
assert np.allclose(_eigh_calls_p24[2], _C_repeat_p24, atol=ATOL, rtol=RTOL)
_evals_repeat_p24 = np.maximum(_ORIGINAL_EIGH_P24(_C_repeat_p24)[0][::-1], 0.0)
assert np.allclose(_repeat_model_p24.explained_variance_, _evals_repeat_p24[:2], atol=ATOL, rtol=RTOL)
assert np.allclose(
    _repeat_model_p24.explained_variance_ratio_,
    _evals_repeat_p24[:2] / _evals_repeat_p24.sum(), atol=ATOL, rtol=RTOL,
)
_P_repeat_p24 = subspace_projector(_repeat_model_p24)
_P_repeat_ref_p24 = np.diag([1.0, 1.0, 0.0, 0.0])
assert np.allclose(_repeat_model_p24.explained_variance_[0], _repeat_model_p24.explained_variance_[1], atol=ATOL, rtol=RTOL)
assert np.allclose(_P_repeat_p24, _P_repeat_ref_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(_P_repeat_p24 @ _P_repeat_p24, _P_repeat_p24, atol=ATOL, rtol=RTOL)
_repeat_hat_p24 = _repeat_model_p24.inverse_transform(_repeat_model_p24.transform(_X_repeat_p24))
assert np.allclose(_repeat_hat_p24, _X_repeat_p24, atol=ATOL, rtol=RTOL)

# Controlled refit: returned eigenpairs must drive all state and both API directions.
_X_refit_p24 = np.array([
    [14.0, -5.0, 7.0, 2.0], [22.0, 4.0, 1.0, -6.0],
    [10.0, 9.0, -4.0, 5.0], [25.0, -1.0, 8.0, 11.0],
    [17.0, 6.0, 13.0, -2.0], [8.0, -8.0, 0.0, 15.0],
])
_mean_refit_p24 = _X_refit_p24.mean(axis=0)
_Xc_refit_p24 = _X_refit_p24 - _mean_refit_p24
_C_refit_p24 = _Xc_refit_p24.T @ _Xc_refit_p24 / (_X_refit_p24.shape[0] - 1)
assert not np.allclose(_C_refit_p24, _C_rank_p24, atol=ATOL, rtol=RTOL)
_sentinel_values_p24 = np.array([6.0, 19.0, 3.5, 10.0])
_sentinel_vectors_p24 = np.array([
    [0.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0, 0.0],
])
_controlled_calls_p24 = []
def _controlled_eigh_p24(matrix):
    _controlled_calls_p24.append(np.array(matrix, copy=True))
    assert np.allclose(matrix, _C_refit_p24, atol=ATOL, rtol=RTOL)
    return _sentinel_values_p24.copy(), _sentinel_vectors_p24.copy()
_X_refit_before_p24 = _X_refit_p24.copy()
np.linalg.eigh = _controlled_eigh_p24
try:
    _returned_refit_p24 = _rank_model_p24.fit(_X_refit_p24)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P24
assert _returned_refit_p24 is _rank_model_p24 and len(_controlled_calls_p24) == 1
assert np.array_equal(_X_refit_p24, _X_refit_before_p24)
_sentinel_order_p24 = np.argsort(_sentinel_values_p24)[::-1]
_expected_values_p24 = _sentinel_values_p24[_sentinel_order_p24]
_expected_scaled_p24 = _expected_values_p24 / _expected_values_p24.max()
_expected_components_p24 = _sentinel_vectors_p24[:, _sentinel_order_p24[:2]].T
assert np.array_equal(_rank_model_p24.mean_, _mean_refit_p24)
assert np.array_equal(_rank_model_p24.explained_variance_, _expected_values_p24[:2])
assert np.array_equal(
    _rank_model_p24.explained_variance_ratio_,
    _expected_scaled_p24[:2] / _expected_scaled_p24.sum(),
)
for _j_controlled_p24 in range(2):
    _P_model_controlled_p24 = np.outer(
        _rank_model_p24.components_[_j_controlled_p24],
        _rank_model_p24.components_[_j_controlled_p24],
    )
    _P_expected_controlled_p24 = np.outer(
        _expected_components_p24[_j_controlled_p24],
        _expected_components_p24[_j_controlled_p24],
    )
    assert np.allclose(
        _P_model_controlled_p24, _P_expected_controlled_p24,
        atol=ATOL, rtol=RTOL,
    )
_probe_X_p24 = np.array([[29.0, -10.0, 16.0, 4.0], [5.0, 12.0, -7.0, 20.0]])
_probe_before_p24 = _probe_X_p24.copy()
_probe_Z_p24 = _rank_model_p24.transform(_probe_X_p24)
_probe_Z_before_p24 = _probe_Z_p24.copy()
assert np.allclose(
    _probe_Z_p24, (_probe_X_p24 - _mean_refit_p24) @ _rank_model_p24.components_.T,
    atol=ATOL, rtol=RTOL,
)
_probe_hat_p24 = _rank_model_p24.inverse_transform(_probe_Z_p24)
assert np.allclose(
    _probe_hat_p24,
    _probe_Z_p24 @ _rank_model_p24.components_ + _mean_refit_p24,
    atol=ATOL, rtol=RTOL,
)
assert np.array_equal(_probe_X_p24, _probe_before_p24)
assert np.array_equal(_probe_Z_p24, _probe_Z_before_p24)

_X_ratio_p24 = np.array([
    [3.0, -1.0, 2.0], [-2.0, 4.0, 1.0], [0.0, -3.0, 5.0],
    [5.0, 2.0, -4.0], [-4.0, -2.0, -1.0], [1.0, 0.0, 3.0],
])
_X_ratio_before_p24 = _X_ratio_p24.copy()
_Xc_ratio_p24 = _X_ratio_p24 - _X_ratio_p24.mean(axis=0)
_C_ratio_p24 = _Xc_ratio_p24.T @ _Xc_ratio_p24 / (_X_ratio_p24.shape[0] - 1)
assert np.isfinite(_X_ratio_p24).all() and np.isfinite(_C_ratio_p24).all()
_huge_values_p24 = np.full(3, 6.4e307)
_ratio_calls_p24 = []
def _huge_finite_eigh_p24(matrix):
    _ratio_calls_p24.append(np.array(matrix, copy=True))
    assert np.allclose(matrix, _C_ratio_p24, atol=ATOL, rtol=RTOL)
    return _huge_values_p24.copy(), np.eye(3)
_ratio_model_p24 = NumpyPCA(3)
np.linalg.eigh = _huge_finite_eigh_p24
try:
    with np.errstate(over="ignore", invalid="ignore"):
        _ratio_returned_p24 = _ratio_model_p24.fit(_X_ratio_p24)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P24
assert _ratio_returned_p24 is _ratio_model_p24 and len(_ratio_calls_p24) == 1
assert np.array_equal(_X_ratio_p24, _X_ratio_before_p24)
for _name_ratio_p24 in ("mean_", "components_", "explained_variance_", "explained_variance_ratio_"):
    assert np.isfinite(getattr(_ratio_model_p24, _name_ratio_p24)).all()
assert np.array_equal(_ratio_model_p24.explained_variance_, _huge_values_p24)
assert np.allclose(_ratio_model_p24.explained_variance_ratio_, np.full(3, 1.0 / 3.0), atol=ATOL, rtol=RTOL)
assert np.isclose(_ratio_model_p24.explained_variance_ratio_.sum(), 1.0, atol=ATOL, rtol=RTOL)

_state_before_failed_refit_p24 = {
    name: getattr(_rank_model_p24, name).copy()
    for name in ("mean_", "components_", "explained_variance_", "explained_variance_ratio_")
}
_scores_before_failed_refit_p24 = _rank_model_p24.transform(_probe_X_p24).copy()
_hat_before_failed_refit_p24 = _rank_model_p24.inverse_transform(_scores_before_failed_refit_p24).copy()
_nonfinite_returns_p24 = (
    (np.array([np.nan, 3.0, 2.0, 1.0]), np.eye(4)),
    (np.array([4.0, 3.0, 2.0, 1.0]), np.array([[1.0, 0.0, 0.0, 0.0], [0.0, np.inf, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]])),
)
for _bad_values_p24, _bad_vectors_p24 in _nonfinite_returns_p24:
    _nonfinite_calls_p24 = []
    def _nonfinite_eigh_p24(matrix):
        _nonfinite_calls_p24.append(np.array(matrix, copy=True))
        assert np.allclose(matrix, _C_rank_p24, atol=ATOL, rtol=RTOL)
        return _bad_values_p24.copy(), _bad_vectors_p24.copy()
    np.linalg.eigh = _nonfinite_eigh_p24
    try:
        _rank_model_p24.fit(_X_rank_p24)
    except ValueError:
        pass
    else:
        raise AssertionError("non-finite eigenpairs must raise ValueError")
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P24
    assert len(_nonfinite_calls_p24) == 1
    for _name_atomic_p24, _old_atomic_p24 in _state_before_failed_refit_p24.items():
        assert np.array_equal(getattr(_rank_model_p24, _name_atomic_p24), _old_atomic_p24)
    _scores_after_failed_refit_p24 = _rank_model_p24.transform(_probe_X_p24)
    assert np.array_equal(_scores_after_failed_refit_p24, _scores_before_failed_refit_p24)
    assert np.array_equal(
        _rank_model_p24.inverse_transform(_scores_after_failed_refit_p24),
        _hat_before_failed_refit_p24,
    )

_theta_p24 = 0.41
_R_p24 = np.array([[np.cos(_theta_p24), -np.sin(_theta_p24)], [np.sin(_theta_p24), np.cos(_theta_p24)]])
_rotated_basis_p24 = _R_p24 @ _repeat_model_p24.components_
assert not np.allclose(_rotated_basis_p24, _repeat_model_p24.components_, atol=ATOL, rtol=RTOL)
assert np.allclose(_rotated_basis_p24.T @ _rotated_basis_p24, _P_repeat_p24, atol=ATOL, rtol=RTOL)

_invalid_fit_p24 = (
    (np.ones(4), 1), (np.ones((1, 4)), 1), (np.ones((3, 0)), 1),
    (np.array([[1.0, np.inf], [2.0, 3.0]]), 1),
    (np.array([[1e200, -1e200, 1e200, -1e200], [-1e200, 1e200, -1e200, 1e200], [0.0, 0.0, 0.0, 0.0]]), 1),
    (np.array([["a", "b"], ["c", "d"]]), 1),
    (np.ones((3, 2)), 0), (np.ones((3, 2)), 3), (np.ones((3, 2)), True),
)
for _X_bad_p24, _k_bad_p24 in _invalid_fit_p24:
    _bad_calls_p24 = []
    def _unexpected_eigh_p24(matrix):
        _bad_calls_p24.append(np.array(matrix, copy=True))
        return _ORIGINAL_EIGH_P24(matrix)
    np.linalg.eigh = _unexpected_eigh_p24
    try:
        with np.errstate(over="ignore", invalid="ignore"):
            NumpyPCA(_k_bad_p24).fit(_X_bad_p24)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid fit input must raise ValueError")
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P24
    assert _bad_calls_p24 == []

for _bad_X_p24 in (
    np.ones(4), np.empty((0, 4)), np.ones((2, 5)),
    np.array([[1.0, 2.0, np.nan, 4.0]]),
):
    try:
        _rank_model_p24.transform(_bad_X_p24)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid transform input must raise ValueError")
for _bad_Z_p24 in (
    np.ones(2), np.empty((0, 2)), np.ones((3, 3)), np.array([[1.0, np.nan]]),
):
    try:
        _rank_model_p24.inverse_transform(_bad_Z_p24)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid inverse-transform input must raise ValueError")

### Answer check

The immutable contract assertions above verify overflow-safe maximum-scaled ratios, atomic fit, and the projector helper across rank-deficient and repeated-eigenvalue regimes, with one-`eigh` behavior and `ATOL = 1e-10`, `RTOL = 0.0` comparisons.